In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
from PIL import Image
from skimage.feature import hog
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.svm import LinearSVC
from sklearn.multioutput import MultiOutputClassifier
from sklearn.metrics import classification_report, f1_score
import matplotlib.pyplot as plt

# Dataset paths
root = Path("../data/processed")

train = pd.read_csv(root / "train_labels.csv")
validation = pd.read_csv(root / "validation_labels.csv")
test = pd.read_csv(root / "test_labels.csv")

products = [
    "B00O4OR4GQ",
    "B013UJS35G",
    "B0159A0LH2",
    "B019D9HESO",
    "B01B1GE6KG",
]

print("Baseline notebook setup complete.")
print("Train:", len(train))
print("Validation:", len(validation))
print("Test:", len(test))

Baseline notebook setup complete.
Train: 1400
Validation: 200
Test: 200


In [3]:
from skimage.feature import hog
from PIL import Image
import numpy as np
from pathlib import Path

IMAGE_SIZE = (64, 64)

def extract_hog_features(filenames, split):
    image_dir = root / split
    features = []

    for i, filename in enumerate(filenames):
        image_path = image_dir / filename

        image = Image.open(image_path).convert("L")
        image = image.resize(IMAGE_SIZE)

        image_array = np.asarray(image)

        feature_vector = hog(
            image_array,
            orientations=9,
            pixels_per_cell=(8, 8),
            cells_per_block=(2, 2),
            block_norm="L2-Hys"
        )

        features.append(feature_vector)

        if (i + 1) % 200 == 0:
            print(f"{split}: processed {i + 1}/{len(filenames)} images")

    return np.array(features, dtype=np.float32)


X_train = extract_hog_features(train["filename"], "train")
X_val = extract_hog_features(validation["filename"], "validation")

# Binary labels: 1 = product present, 0 = product absent
y_train = (train[[f"qty_{p}" for p in products]] > 0).astype(int)
y_val = (validation[[f"qty_{p}" for p in products]] > 0).astype(int)

print("\nFeature extraction complete.")
print("Training features:", X_train.shape)
print("Validation features:", X_val.shape)
print("Training labels:", y_train.shape)
print("Validation labels:", y_val.shape)

train: processed 200/1400 images
train: processed 400/1400 images
train: processed 600/1400 images
train: processed 800/1400 images
train: processed 1000/1400 images
train: processed 1200/1400 images
train: processed 1400/1400 images
validation: processed 200/200 images

Feature extraction complete.
Training features: (1400, 1764)
Validation features: (200, 1764)
Training labels: (1400, 5)
Validation labels: (200, 5)


In [4]:
from sklearn.svm import LinearSVC
from sklearn.multioutput import MultiOutputClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, f1_score
import pandas as pd

# One binary classifier per catalogue product
baseline = MultiOutputClassifier(
    make_pipeline(
        StandardScaler(),
        LinearSVC(class_weight="balanced", random_state=42, max_iter=5000)
    )
)

print("Training baseline model...")
baseline.fit(X_train, y_train)

print("Baseline training complete!")

# Evaluate on validation data only
y_val_pred = baseline.predict(X_val)

print("\nValidation results by product:\n")

for i, asin in enumerate(products):
    print(f"\nProduct: {asin}")
    print(
        classification_report(
            y_val.iloc[:, i],
            y_val_pred[:, i],
            labels=[0, 1],
            target_names=["Absent", "Present"],
            zero_division=0
        )
    )

print(
    "Overall micro F1:",
    round(f1_score(y_val, y_val_pred, average="micro", zero_division=0), 4)
)

print(
    "Overall macro F1:",
    round(f1_score(y_val, y_val_pred, average="macro", zero_division=0), 4)
)

Training baseline model...
Baseline training complete!

Validation results by product:


Product: B00O4OR4GQ
              precision    recall  f1-score   support

      Absent       0.92      0.83      0.87       175
     Present       0.30      0.52      0.38        25

    accuracy                           0.79       200
   macro avg       0.61      0.67      0.63       200
weighted avg       0.85      0.79      0.81       200


Product: B013UJS35G
              precision    recall  f1-score   support

      Absent       0.92      0.91      0.92       155
     Present       0.70      0.73      0.72        45

    accuracy                           0.87       200
   macro avg       0.81      0.82      0.82       200
weighted avg       0.87      0.87      0.87       200


Product: B0159A0LH2
              precision    recall  f1-score   support

      Absent       0.88      0.88      0.88       167
     Present       0.38      0.36      0.37        33

    accuracy                   

In [6]:
from pathlib import Path
import joblib

save_path = Path("../models")
save_path.mkdir(parents=True, exist_ok=True)

joblib.dump(baseline, save_path / "hog_svm_baseline.joblib")

print("Baseline model saved successfully.")

Baseline model saved successfully.


In [8]:
import torch
from torch import nn
from torchvision import models, transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from pathlib import Path
import numpy as np
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

products = [
    "B00O4OR4GQ",
    "B013UJS35G",
    "B0159A0LH2",
    "B019D9HESO",
    "B01B1GE6KG",
]

root = Path("../data/processed")
val = pd.read_csv(root / "validation_labels.csv")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class BinDataset(Dataset):
    def __init__(self, dataframe, split, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = root / split
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        with Image.open(self.image_dir / row["filename"]) as img:
            image = self.transform(img.convert("RGB"))
        labels = (
            row[[f"qty_{p}" for p in products]]
            .to_numpy(dtype=np.float32) > 0
        ).astype(np.float32)
        return image, torch.tensor(labels)

val_dataset = BinDataset(val, "validation", transform)
val_loader = DataLoader(
    val_dataset, batch_size=16, shuffle=False, num_workers=0
)

model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(products))
model = model.to(device)

cnn_path = Path("../models/resnet18_finetuned_best.pth")
model.load_state_dict(torch.load(cnn_path, map_location=device))
model.eval()

print("CNN checkpoint loaded.")
print("Device:", device)
print("Validation images:", len(val_dataset))

CNN checkpoint loaded.
Device: cuda
Validation images: 200


In [9]:
from sklearn.metrics import f1_score
import numpy as np
import pandas as pd
import torch

# HOG + SVM predictions
baseline_preds = baseline.predict(X_val)

# CNN predictions
cnn_probs = []

with torch.no_grad():
    for images, labels in val_loader:
        images = images.to(device)
        probs = torch.sigmoid(model(images))
        cnn_probs.extend(probs.cpu().numpy())

cnn_preds = (np.array(cnn_probs) >= 0.5).astype(int)

# Ground-truth labels: product present or absent
y_true = (y_val.to_numpy() > 0).astype(int)

results = pd.DataFrame({
    "Model": ["HOG + SVM", "Fine-tuned ResNet-18"],
    "Macro F1": [
        f1_score(y_true, baseline_preds, average="macro", zero_division=0),
        f1_score(y_true, cnn_preds, average="macro", zero_division=0)
    ],
    "Micro F1": [
        f1_score(y_true, baseline_preds, average="micro", zero_division=0),
        f1_score(y_true, cnn_preds, average="micro", zero_division=0)
    ]
})

print(results.round(4).to_string(index=False))

               Model  Macro F1  Micro F1
           HOG + SVM    0.4053    0.4431
Fine-tuned ResNet-18    0.8601    0.8723


In [10]:
import pandas as pd
from pathlib import Path

root = Path("../data/processed")

train_labels = pd.read_csv(root / "train_labels.csv")
val_labels = pd.read_csv(root / "validation_labels.csv")

quantity_columns = [f"qty_{p}" for p in products]

print("Training quantity distribution:")
for col in quantity_columns:
    print(f"\n{col}")
    print(train_labels[col].value_counts().sort_index().to_dict())

print("\nValidation quantity distribution:")
for col in quantity_columns:
    print(f"\n{col}")
    print(val_labels[col].value_counts().sort_index().to_dict())

Training quantity distribution:

qty_B00O4OR4GQ
{0: 1190, 1: 130, 2: 61, 3: 10, 4: 8, 5: 1}

qty_B013UJS35G
{0: 1050, 1: 330, 2: 18, 3: 2}

qty_B0159A0LH2
{0: 1231, 1: 165, 2: 4}

qty_B019D9HESO
{0: 1227, 1: 117, 2: 43, 3: 11, 4: 2}

qty_B01B1GE6KG
{0: 1232, 1: 116, 2: 47, 3: 5}

Validation quantity distribution:

qty_B00O4OR4GQ
{0: 175, 1: 15, 2: 4, 3: 4, 4: 2}

qty_B013UJS35G
{0: 155, 1: 45}

qty_B0159A0LH2
{0: 167, 1: 33}

qty_B019D9HESO
{0: 172, 1: 20, 2: 7, 4: 1}

qty_B01B1GE6KG
{0: 181, 1: 11, 2: 7, 3: 1}


In [11]:
# Load the best fine-tuned CNN
model.load_state_dict(
    torch.load(cnn_path, map_location=device)
)
model.eval()

# Remove the final classification layer to get image features
feature_extractor = torch.nn.Sequential(
    *list(model.children())[:-1]
).to(device)
feature_extractor.eval()

def extract_features(loader):
    features = []
    quantities = []

    with torch.no_grad():
        for images, labels in loader:
            images = images.to(device)

            batch_features = feature_extractor(images)
            batch_features = batch_features.flatten(1)

            features.append(batch_features.cpu().numpy())
            quantities.append(labels.numpy())

    return np.vstack(features), np.vstack(quantities)

# Dataset labels here must be actual quantities, not binary presence labels.

In [12]:
class QuantityBinDataset(Dataset):
    def __init__(self, dataframe, split, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = root / split
        self.transform = transform
        self.label_columns = [f"qty_{p}" for p in products]

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        with Image.open(self.image_dir / row["filename"]) as img:
            image = self.transform(img.convert("RGB"))

        # Keep actual quantities: 0, 1, 2, 3, ...
        quantities = row[self.label_columns].to_numpy(dtype=np.float32)

        return image, torch.tensor(quantities)

train_quantity_dataset = QuantityBinDataset(
    train_labels, "train", transform
)
val_quantity_dataset = QuantityBinDataset(
    val_labels, "validation", transform
)

train_quantity_loader = DataLoader(
    train_quantity_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)
val_quantity_loader = DataLoader(
    val_quantity_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

print("Quantity-aware loaders ready.")
print("Training images:", len(train_quantity_dataset))
print("Validation images:", len(val_quantity_dataset))

Quantity-aware loaders ready.
Training images: 1400
Validation images: 200


In [13]:
# Extract features from the trained ResNet-18
model.load_state_dict(
    torch.load(cnn_path, map_location=device)
)
model.eval()

feature_extractor = torch.nn.Sequential(
    *list(model.children())[:-1]
).to(device)
feature_extractor.eval()

def extract_features(loader):
    all_features = []
    all_quantities = []

    with torch.no_grad():
        for images, quantities in loader:
            images = images.to(device)

            features = feature_extractor(images).flatten(1)

            all_features.append(features.cpu().numpy())
            all_quantities.append(quantities.numpy())

    return np.vstack(all_features), np.vstack(all_quantities)

X_train_features, y_train_quantities = extract_features(
    train_quantity_loader
)
X_val_features, y_val_quantities = extract_features(
    val_quantity_loader
)

print("Training features:", X_train_features.shape)
print("Training quantities:", y_train_quantities.shape)
print("Validation features:", X_val_features.shape)
print("Validation quantities:", y_val_quantities.shape)

Training features: (1400, 512)
Training quantities: (1400, 5)
Validation features: (200, 512)
Validation quantities: (200, 5)


In [14]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error, accuracy_score
from sklearn.multioutput import MultiOutputClassifier
import numpy as np
import pandas as pd

# One quantity classifier per product
quantity_model = MultiOutputClassifier(
    RandomForestClassifier(
        n_estimators=200,
        class_weight="balanced",
        random_state=42,
        min_samples_leaf=2,
        n_jobs=-1
    )
)

print("Training quantity predictors...")

quantity_model.fit(
    X_train_features,
    y_train_quantities.astype(int)
)

print("Quantity training complete!")

# Predict quantities on validation data
quantity_preds = quantity_model.predict(X_val_features).astype(int)

# Metrics
exact_match = np.mean(
    np.all(quantity_preds == y_val_quantities.astype(int), axis=1)
)
mae = mean_absolute_error(
    y_val_quantities.astype(int),
    quantity_preds
)

print(f"\nOverall quantity exact-match accuracy: {exact_match:.4f}")
print(f"Overall quantity MAE: {mae:.4f}")

print("\nPer-product results:")
for i, asin in enumerate(products):
    true_q = y_val_quantities[:, i].astype(int)
    pred_q = quantity_preds[:, i]

    print(
        f"{asin} | "
        f"Exact accuracy: {accuracy_score(true_q, pred_q):.4f} | "
        f"MAE: {mean_absolute_error(true_q, pred_q):.4f}"
    )

Training quantity predictors...
Quantity training complete!

Overall quantity exact-match accuracy: 0.6750
Overall quantity MAE: 0.0860

Per-product results:
B00O4OR4GQ | Exact accuracy: 0.9350 | MAE: 0.1150
B013UJS35G | Exact accuracy: 0.9750 | MAE: 0.0250
B0159A0LH2 | Exact accuracy: 0.9350 | MAE: 0.0650
B019D9HESO | Exact accuracy: 0.8900 | MAE: 0.1350
B01B1GE6KG | Exact accuracy: 0.9300 | MAE: 0.0900


In [15]:
import numpy as np
import pandas as pd
from sklearn.metrics import mean_absolute_error

true_quantities = y_val_quantities.astype(int)
pred_quantities = quantity_preds.astype(int)

rows = []

for i, asin in enumerate(products):
    actual = true_quantities[:, i]
    predicted = pred_quantities[:, i]

    present_mask = actual > 0

    rows.append({
        "ASIN": asin,
        "Actual present examples": int(present_mask.sum()),
        "Correct quantity when present": int(
            (actual[present_mask] == predicted[present_mask]).sum()
        ),
        "Exact accuracy when present": (
            float(np.mean(actual[present_mask] == predicted[present_mask]))
            if present_mask.any() else np.nan
        ),
        "MAE when present": (
            float(mean_absolute_error(
                actual[present_mask], predicted[present_mask]
            ))
            if present_mask.any() else np.nan
        )
    })

print(pd.DataFrame(rows).round(4).to_string(index=False))

      ASIN  Actual present examples  Correct quantity when present  Exact accuracy when present  MAE when present
B00O4OR4GQ                       25                             12                       0.4800            0.9200
B013UJS35G                       45                             42                       0.9333            0.0667
B0159A0LH2                       33                             22                       0.6667            0.3333
B019D9HESO                       28                              8                       0.2857            0.8929
B01B1GE6KG                       19                              7                       0.3684            0.8421


In [16]:
for i, asin in enumerate(products):
    actual = true_quantities[:, i]
    predicted = pred_quantities[:, i]
    present = actual > 0

    print(f"\n{asin}")
    print("Actual quantities when present:")
    print(pd.Series(actual[present]).value_counts().sort_index().to_dict())

    print("Predictions for those same examples:")
    print(pd.Series(predicted[present]).value_counts().sort_index().to_dict())

    print(
        "Predicted zero despite actual presence:",
        int((predicted[present] == 0).sum())
    )


B00O4OR4GQ
Actual quantities when present:
{1: 15, 2: 4, 3: 4, 4: 2}
Predictions for those same examples:
{0: 7, 1: 16, 2: 2}
Predicted zero despite actual presence: 7

B013UJS35G
Actual quantities when present:
{1: 45}
Predictions for those same examples:
{0: 3, 1: 42}
Predicted zero despite actual presence: 3

B0159A0LH2
Actual quantities when present:
{1: 33}
Predictions for those same examples:
{0: 11, 1: 22}
Predicted zero despite actual presence: 11

B019D9HESO
Actual quantities when present:
{1: 20, 2: 7, 4: 1}
Predictions for those same examples:
{0: 15, 1: 13}
Predicted zero despite actual presence: 15

B01B1GE6KG
Actual quantities when present:
{1: 11, 2: 7, 3: 1}
Predictions for those same examples:
{0: 7, 1: 12}
Predicted zero despite actual presence: 7


In [17]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import mean_absolute_error
from collections import Counter

positive_quantity_models = {}

for i, asin in enumerate(products):
    quantities = y_train_quantities[:, i].astype(int)
    present_mask = quantities > 0

    X_positive = X_train_features[present_mask]
    y_positive = quantities[present_mask]

    print(f"\n{asin}")
    print("Training examples with product present:", len(y_positive))
    print("Quantity distribution:", Counter(y_positive))

    model_positive = RandomForestClassifier(
        n_estimators=200,
        min_samples_leaf=1,
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    )

    model_positive.fit(X_positive, y_positive)
    positive_quantity_models[asin] = model_positive

print("\nAll five positive-quantity models trained.")


B00O4OR4GQ
Training examples with product present: 210
Quantity distribution: Counter({np.int64(1): 130, np.int64(2): 61, np.int64(3): 10, np.int64(4): 8, np.int64(5): 1})

B013UJS35G
Training examples with product present: 350
Quantity distribution: Counter({np.int64(1): 330, np.int64(2): 18, np.int64(3): 2})

B0159A0LH2
Training examples with product present: 169
Quantity distribution: Counter({np.int64(1): 165, np.int64(2): 4})

B019D9HESO
Training examples with product present: 173
Quantity distribution: Counter({np.int64(1): 117, np.int64(2): 43, np.int64(3): 11, np.int64(4): 2})

B01B1GE6KG
Training examples with product present: 168
Quantity distribution: Counter({np.int64(1): 116, np.int64(2): 47, np.int64(3): 5})

All five positive-quantity models trained.


In [18]:
from sklearn.metrics import accuracy_score, mean_absolute_error
import pandas as pd
import numpy as np

print("POSITIVE-QUANTITY MODEL EVALUATION")
print("=" * 65)

for i, asin in enumerate(products):
    actual = y_val_quantities[:, i].astype(int)
    present_mask = actual > 0

    X_present = X_val_features[present_mask]
    y_present = actual[present_mask]

    predicted = positive_quantity_models[asin].predict(X_present)

    print(f"\n{asin}")
    print("Validation examples where product is present:", len(y_present))
    print("Actual quantities:", pd.Series(y_present).value_counts().sort_index().to_dict())
    print("Predicted quantities:", pd.Series(predicted).value_counts().sort_index().to_dict())
    print(f"Quantity exact accuracy: {accuracy_score(y_present, predicted):.4f}")
    print(f"Quantity MAE: {mean_absolute_error(y_present, predicted):.4f}")

POSITIVE-QUANTITY MODEL EVALUATION

B00O4OR4GQ
Validation examples where product is present: 25
Actual quantities: {1: 15, 2: 4, 3: 4, 4: 2}
Predicted quantities: {1: 21, 2: 4}
Quantity exact accuracy: 0.6400
Quantity MAE: 0.6400

B013UJS35G
Validation examples where product is present: 45
Actual quantities: {1: 45}
Predicted quantities: {1: 45}
Quantity exact accuracy: 1.0000
Quantity MAE: 0.0000

B0159A0LH2
Validation examples where product is present: 33
Actual quantities: {1: 33}
Predicted quantities: {1: 33}
Quantity exact accuracy: 1.0000
Quantity MAE: 0.0000

B019D9HESO
Validation examples where product is present: 28
Actual quantities: {1: 20, 2: 7, 4: 1}
Predicted quantities: {1: 28}
Quantity exact accuracy: 0.7143
Quantity MAE: 0.3571

B01B1GE6KG
Validation examples where product is present: 19
Actual quantities: {1: 11, 2: 7, 3: 1}
Predicted quantities: {1: 16, 2: 3}
Quantity exact accuracy: 0.7368
Quantity MAE: 0.3158


In [19]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, mean_absolute_error

# Get ResNet-18 product-presence predictions
model.eval()
presence_predictions = []

with torch.no_grad():
    for images, _ in val_loader:
        images = images.to(device)
        logits = model(images)
        predictions = (torch.sigmoid(logits) >= 0.5).int()
        presence_predictions.append(predictions.cpu().numpy())

presence_predictions = np.vstack(presence_predictions)

# Combine presence and positive-quantity predictions
true_quantities = y_val_quantities.astype(int)
combined_predictions = np.zeros_like(true_quantities, dtype=int)

for i, asin in enumerate(products):
    predicted_present = presence_predictions[:, i] == 1
    combined_predictions[predicted_present, i] = (
        positive_quantity_models[asin].predict(
            X_val_features[predicted_present]
        )
    )

# Overall evaluation
exact_match = np.mean(
    np.all(combined_predictions == true_quantities, axis=1)
)
overall_mae = mean_absolute_error(
    true_quantities, combined_predictions
)

print(f"Overall exact-match accuracy: {exact_match:.4f}")
print(f"Overall quantity MAE: {overall_mae:.4f}")

print("\nPer-product results:")
for i, asin in enumerate(products):
    actual = true_quantities[:, i]
    predicted = combined_predictions[:, i]

    print(
        f"{asin}: "
        f"Exact accuracy={accuracy_score(actual, predicted):.4f}, "
        f"MAE={mean_absolute_error(actual, predicted):.4f}"
    )

Overall exact-match accuracy: 0.7500
Overall quantity MAE: 0.0670

Per-product results:
B00O4OR4GQ: Exact accuracy=0.9450, MAE=0.1000
B013UJS35G: Exact accuracy=0.9850, MAE=0.0150
B0159A0LH2: Exact accuracy=0.9400, MAE=0.0600
B019D9HESO: Exact accuracy=0.9050, MAE=0.1100
B01B1GE6KG: Exact accuracy=0.9600, MAE=0.0500


In [20]:
from pathlib import Path
import joblib

models_dir = Path("../models")
models_dir.mkdir(parents=True, exist_ok=True)

joblib.dump(
    positive_quantity_models,
    models_dir / "positive_quantity_models.joblib"
)

print("Positive-quantity models saved successfully.")

Positive-quantity models saved successfully.


In [21]:
import joblib
import pandas as pd
import torch
from pathlib import Path
from torchvision import models, transforms
from torch import nn

project_root = Path(r"D:\Smart bin classifier")
models_dir = project_root / "models"
data_root = project_root / "data" / "processed"

products = [
    "B00O4OR4GQ",
    "B013UJS35G",
    "B0159A0LH2",
    "B019D9HESO",
    "B01B1GE6KG",
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the saved ResNet-18 model
model = models.resnet18(weights=None)
model.fc = nn.Linear(model.fc.in_features, len(products))
model.load_state_dict(
    torch.load(
        models_dir / "resnet18_finetuned_best.pth",
        map_location=device,
        weights_only=True
    )
)
model = model.to(device)
model.eval()

# Load the positive-quantity models
positive_quantity_models = joblib.load(
    models_dir / "positive_quantity_models.joblib"
)

# Load test labels for evaluation only
test_labels = pd.read_csv(data_root / "test_labels.csv")

print("Device:", device)
print("Test images:", len(list((data_root / "test").glob("*"))))
print("Test label rows:", len(test_labels))
print("Models loaded successfully.")

Device: cuda
Test images: 200
Test label rows: 200
Models loaded successfully.


In [22]:
import numpy as np
from PIL import Image
from torch.utils.data import Dataset, DataLoader

# Image preprocessing must match training
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],
        std=[0.229, 0.224, 0.225]
    )
])

class TestImageDataset(Dataset):
    def __init__(self, filenames):
        self.filenames = filenames
        self.image_dir = data_root / "test"

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, index):
        filename = self.filenames[index]
        with Image.open(self.image_dir / filename) as img:
            image = test_transform(img.convert("RGB"))
        return image

test_dataset = TestImageDataset(test_labels["filename"].tolist())
test_loader = DataLoader(
    test_dataset,
    batch_size=16,
    shuffle=False,
    num_workers=0
)

# Predict product presence using ResNet-18
presence_batches = []

model.eval()
with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        logits = model(images)
        predictions = (torch.sigmoid(logits) >= 0.5).int()
        presence_batches.append(predictions.cpu().numpy())

test_presence_predictions = np.vstack(presence_batches)

# Extract image features for quantity prediction
feature_extractor = torch.nn.Sequential(
    *list(model.children())[:-1]
).to(device)
feature_extractor.eval()

feature_batches = []

with torch.no_grad():
    for images in test_loader:
        images = images.to(device)
        features = feature_extractor(images).flatten(1)
        feature_batches.append(features.cpu().numpy())

X_test_features = np.vstack(feature_batches)

# Combine presence and positive-quantity predictions
test_quantity_predictions = np.zeros(
    (len(test_labels), len(products)),
    dtype=int
)

for i, asin in enumerate(products):
    present_mask = test_presence_predictions[:, i] == 1

    if present_mask.any():
        test_quantity_predictions[present_mask, i] = (
            positive_quantity_models[asin].predict(
                X_test_features[present_mask]
            )
        )

print("Test images processed:", len(test_quantity_predictions))
print("Presence predictions shape:", test_presence_predictions.shape)
print("Quantity predictions shape:", test_quantity_predictions.shape)
print("Image-only predictions generated successfully.")

Test images processed: 200
Presence predictions shape: (200, 5)
Quantity predictions shape: (200, 5)
Image-only predictions generated successfully.


In [23]:
from sklearn.metrics import accuracy_score, mean_absolute_error
import numpy as np
import pandas as pd

# Ground-truth quantities are used only for evaluation
test_quantity_columns = [f"qty_{asin}" for asin in products]
y_test_true = test_labels[test_quantity_columns].to_numpy(dtype=int)
y_test_pred = test_quantity_predictions

# Overall metrics
exact_match = np.mean(
    np.all(y_test_true == y_test_pred, axis=1)
)
overall_mae = mean_absolute_error(y_test_true, y_test_pred)

print("FINAL TEST RESULTS")
print("=" * 55)
print(f"Test bins: {len(y_test_true)}")
print(f"Exact-match accuracy: {exact_match:.4f}")
print(f"Quantity MAE: {overall_mae:.4f}")

print("\nPer-product results:")
for i, asin in enumerate(products):
    print(
        f"{asin}: "
        f"Exact accuracy={accuracy_score(y_test_true[:, i], y_test_pred[:, i]):.4f}, "
        f"MAE={mean_absolute_error(y_test_true[:, i], y_test_pred[:, i]):.4f}"
    )

FINAL TEST RESULTS
Test bins: 200
Exact-match accuracy: 0.6300
Quantity MAE: 0.0950

Per-product results:
B00O4OR4GQ: Exact accuracy=0.8650, MAE=0.2000
B013UJS35G: Exact accuracy=0.9550, MAE=0.0450
B0159A0LH2: Exact accuracy=0.9650, MAE=0.0450
B019D9HESO: Exact accuracy=0.9050, MAE=0.1000
B01B1GE6KG: Exact accuracy=0.9300, MAE=0.0850


In [24]:
import numpy as np

test_has_catalogue_product = (y_test_true > 0).any(axis=1)

print("Total test bins:", len(y_test_true))
print("Bins with at least one catalogue product:",
      int(test_has_catalogue_product.sum()))
print("Bins with no catalogue products:",
      int((~test_has_catalogue_product).sum()))

Total test bins: 200
Bins with at least one catalogue product: 153
Bins with no catalogue products: 47


In [25]:
from sklearn.metrics import confusion_matrix, classification_report
import numpy as np
import pandas as pd

# Only bins with at least one catalogue product can form a meaningful PASS order
eligible_indices = np.where((y_test_true > 0).any(axis=1))[0]

actual_decisions = []
predicted_decisions = []
order_results = []

for idx in eligible_indices:
    true_qty = y_test_true[idx]
    pred_qty = y_test_pred[idx]

    # PASS order: request the products and quantities actually in the bin
    pass_order = {
        products[j]: int(true_qty[j])
        for j in range(len(products))
        if true_qty[j] > 0
    }

    actual_pass = all(
        pred_qty[products.index(asin)] == qty
        for asin, qty in pass_order.items()
    )
    predicted_decisions.append("PASS" if actual_pass else "FAIL")
    actual_decisions.append("PASS")

    order_results.append({
        "test_index": int(idx),
        "order_type": "PASS",
        "actual_order": "PASS",
        "predicted_order": "PASS" if actual_pass else "FAIL"
    })

    # FAIL order: alter the quantity of one product known to be present
    chosen = next(j for j in range(len(products)) if true_qty[j] > 0)
    fail_order_qty = int(true_qty[chosen]) + 1

    actual_fail = pred_qty[chosen] == fail_order_qty
    predicted_decisions.append("PASS" if actual_fail else "FAIL")
    actual_decisions.append("FAIL")

    order_results.append({
        "test_index": int(idx),
        "order_type": "FAIL",
        "actual_order": "FAIL",
        "predicted_order": "PASS" if actual_fail else "FAIL"
    })

# Confusion matrix order: actual labels [FAIL, PASS]
labels = ["FAIL", "PASS"]
cm = confusion_matrix(actual_decisions, predicted_decisions, labels=labels)

print("Eligible test bins:", len(eligible_indices))
print("Orders evaluated:", len(order_results))
print("\nConfusion matrix (rows=actual, columns=predicted)")
print(pd.DataFrame(
    cm,
    index=[f"Actual {x}" for x in labels],
    columns=[f"Predicted {x}" for x in labels]
))

print("\nClassification report:")
print(classification_report(
    actual_decisions,
    predicted_decisions,
    labels=labels,
    zero_division=0
))

print("Warehouse decision accuracy:",
      round(np.mean(np.array(actual_decisions) == np.array(predicted_decisions)), 4))

Eligible test bins: 153
Orders evaluated: 306

Confusion matrix (rows=actual, columns=predicted)
             Predicted FAIL  Predicted PASS
Actual FAIL             148               5
Actual PASS              56              97

Classification report:
              precision    recall  f1-score   support

        FAIL       0.73      0.97      0.83       153
        PASS       0.95      0.63      0.76       153

    accuracy                           0.80       306
   macro avg       0.84      0.80      0.79       306
weighted avg       0.84      0.80      0.79       306

Warehouse decision accuracy: 0.8007
